# Sección 9 — Enfoque B: $\mathcal{G}(p,c)$ cruzado por k-pairs

Implementa la fórmula definida en el MD 9:
$$\mathcal{G}(p,\,c) = \left(\bigcup_{k=1}^{4} \mathcal{G}_k(p,c)\right) \setminus \{\emptyset\}$$

El objetivo es recuperar el GUID correcto para cada uno de los **36 eventos centinela k=1**,
consultando los cuatro k-pairs en lugar de solo k=1 (Enfoque A).

## Setup

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_colwidth', 80)

NULL_GUID = '00000000-0000-0000-0000-000000000000'
DATA_DIR  = '../dataset/run-01-apt-1/'
print('OK')

In [ ]:
df = pd.read_csv(DATA_DIR + '02_sysmon-run-01.csv', low_memory=False)
df['_original_row_index'] = df.index
df['ts'] = pd.to_datetime(df['timestamp'].where(df['timestamp'] > 0), unit='ms')

viol = pd.read_csv(DATA_DIR + '04_sysmon-run-01-violations.csv', low_memory=False)
viol['ts'] = pd.to_datetime(viol['timestamp'].where(viol['timestamp'] > 0), unit='ms')

sentinel_k1 = (
    viol[~viol['EventID'].isin([8, 10]) & (viol['ProcessGuid'] == NULL_GUID)]
    .sort_values('_original_row_index')
    .reset_index(drop=True)
)

print(f'df          : {len(df):,} filas')
print(f'sentinel_k1 : {len(sentinel_k1)} eventos')

---
## Tabla resumen: $\lvert\mathcal{G}(p,c)\rvert$ para los 36 eventos centinela k=1

Calculamos $\mathcal{G}(p,c)$ para cada uno de los 36 eventos y clasificamos
la acción de recuperación. Analizaremos los casos **evento por evento**,
comenzando por los de $\lvert\mathcal{G}\rvert = 1$ (regla `REPLACE_GUID` con
verificación temporal), luego los de $\lvert\mathcal{G}\rvert > 1$ (`REVIEW`),
y finalmente los de $\lvert\mathcal{G}\rvert = 0$ (`BOOT_ARTIFACT`) si los hubiera.

In [ ]:
def compute_G(df, p, c):
    """Retorna G(p,c): conjunto de GUIDs reales observados para (p,c) en los 4 k-pairs."""
    g1 = set(df[~df['EventID'].isin([8,10]) & (df['Computer']==c) & (df['ProcessId']==p)
              ]['ProcessGuid'].dropna()) - {NULL_GUID}
    g2 = set(df[(df['EventID']==1) & (df['Computer']==c) & (df['ParentProcessId']==p)
              ]['ParentProcessGuid'].dropna()) - {NULL_GUID}
    g3 = set(df[df['EventID'].isin([8,10]) & (df['Computer']==c) & (df['SourceProcessId']==p)
              ]['SourceProcessGUID'].dropna()) - {NULL_GUID}
    g4 = set(df[df['EventID'].isin([8,10]) & (df['Computer']==c) & (df['TargetProcessId']==p)
              ]['TargetProcessGUID'].dropna()) - {NULL_GUID}
    return g1 | g2 | g3 | g4

rows = []
for i, row in sentinel_k1.iterrows():
    G = compute_G(df, row['ProcessId'], row['Computer'])
    card = len(G)
    accion = 'REPLACE_GUID*' if card == 1 else ('BOOT_ARTIFACT' if card == 0 else 'REVIEW')
    rows.append({
        'ev'      : i,
        'row_csv' : int(row['_original_row_index']),
        'Computer': row['Computer'],
        'PID'     : int(row['ProcessId']),
        'Image'   : row['Image'],
        'ts'      : row['ts'],
        '|G|'     : card,
        'G'       : str(sorted(G)),
        'accion'  : accion,
    })

resumen = pd.DataFrame(rows)
print('Distribución de acciones (antes de verificación temporal):')
print(resumen['accion'].value_counts())
print()
resumen[['ev','row_csv','Computer','PID','Image','ts','|G|','accion']]

---
## Caso $\lvert\mathcal{G}\rvert = 1$ — Evento 04

**PID 2968 · `endofroad.boombox.local` · fila 19619**  
`C:\Windows\System32\conhost.exe`

Primer evento centinela k=1 con exactamente un GUID real observado.
Aplicamos la regla completa incluyendo la verificación temporal:

$$
\text{acción}(e^*, g_0) =
\begin{cases}
\texttt{REPLACE\_GUID} & \text{si } t_{\min}(g_0) \leq t^* \leq t_{\max}(g_0) \\
\texttt{REVIEW}        & \text{si } t^* > t_{\max}(g_0)
\end{cases}
$$

In [ ]:
# Evento centinela e*
ev04 = sentinel_k1.iloc[4]
t_star = ev04['ts']
p04    = ev04['ProcessId']
c04    = ev04['Computer']

print('=== Evento centinela e* ===')
print(ev04[['_original_row_index','EventID','Computer','ProcessId','Image','ts']].to_string())

# G(p, c)
G04 = compute_G(df, p04, c04)
g0  = list(G04)[0]
print(f'\nG({int(p04)}, {c04}) = {{{g0}}}')
print(f'|G| = {len(G04)}  →  candidato único: {g0}')

In [ ]:
# Ciclo de vida de g0: todos sus eventos en el CSV completo
g0_events = (
    df[df['ProcessGuid'] == g0]
    .sort_values('ts')
    [['ts','EventID','ProcessGuid','ProcessId','Image','Computer']]
)

t_min_g0 = g0_events['ts'].min()
t_max_g0 = g0_events['ts'].max()

print(f't_min(g0) = {t_min_g0}')
print(f't*        = {t_star}')
print(f't_max(g0) = {t_max_g0}')
print()

dentro = t_min_g0 <= t_star <= t_max_g0
accion = 'REPLACE_GUID' if dentro else 'REVIEW (t* fuera del ciclo de vida de g0)'
print(f't_min <= t* <= t_max : {dentro}')
print(f'Acción final         : {accion}')
print()
print(f'Eventos de g0 ({len(g0_events)} total):')
g0_events